In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import silhouette_score
import joblib
import seaborn as sns

%load_ext autoreload
%autoreload 2

Algorithm: K-Means clustering (unsupervised) to group research centers into three quality tiers (Premium, Standard, Basic) based on infrastructure and healthcare access features.

In [ ]:
filename = 'research_centers.csv'

# Load

In [ ]:
raw_df = pd.read_csv(filename)
raw_df.head()

In [ ]:
raw_df.info(), raw_df.duplicated().sum()

# Explore

In [ ]:
raw_df['researchCenterName'].nunique(), raw_df['city'].nunique()

### Unique Centers per city

In [ ]:
city_counts = raw_df.groupby('city')['researchCenterId'].nunique().sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.tab20.colors  # or `tab10` etc
ax = city_counts.plot(
    kind='barh',
    legend=False,
    ax=ax,
    grid=True,
    color=[colors[i % len(colors)] for i in range(len(city_counts))],
    width=0.8
)

for i, value in enumerate(city_counts):
    ax.text(value + 0.1, i, str(value), va='center')

ax.set_title('Unique Research Centers per City')
ax.set_xlabel('Unique researchCenterId count')
ax.set_ylabel('City')
plt.tight_layout()
plt.show()

### Internal Facilities Count

In [ ]:
city_facilities_df = raw_df.groupby('city')['internalFacilitiesCount'].agg(['mean', 'min', 'max', 'sum', 'count', 'nunique'])
max_bin_count = int(city_facilities_df['max'].max())
city_facilities_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

raw_df['internalFacilitiesCount'].hist(bins=max_bin_count, ax=ax, grid=True)
ax.set_title(f'Distribution of Internal Facilities Count - Total is {raw_df["internalFacilitiesCount"].sum()}')
ax.set_xlabel('Internal Facilities Count')
ax.set_ylabel('Frequency')
ax.set_xticks(range(raw_df['internalFacilitiesCount'].min(), raw_df['internalFacilitiesCount'].max() + 1))
ax.set_xlim(raw_df['internalFacilitiesCount'].min() - 0.5, raw_df['internalFacilitiesCount'].max() + 0.5)
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(raw_df['internalFacilitiesCount'].dropna(), bins=max_bin_count, color='skyblue', edgecolor='black')
plt.title('Histogram of internalFacilitiesCount')
plt.xlabel('internalFacilitiesCount')
plt.ylabel('Frequency')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
cities = raw_df['city'].sort_values().unique()

fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=False)
axes = axes.flatten()

for i, city in enumerate(cities):
    subset = raw_df.loc[raw_df['city'] == city, 'internalFacilitiesCount']
    subset.plot(
        kind='hist',
        bins=max_bin_count,
        ax=axes[i],
        edgecolor='black',
        alpha=0.7,
        title=city
    )
    axes[i].set_xlabel('internalFacilitiesCount')
    axes[i].set_ylabel('Frequency')

for j in range(len(cities), len(axes)):
    axes[j].axis('off')

fig.suptitle('internalFacilitiesCount Distribution by City (3x2 grid)', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### Hospital vs pharmacies

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(raw_df['hospitals_10km'], raw_df['pharmacies_10km'], c=raw_df['internalFacilitiesCount'], cmap='viridis', alpha=0.75)
plt.colorbar(scatter, label='internalFacilitiesCount')
plt.title('Hospitals vs Pharmacies (color=internalFacilitiesCount)')
plt.xlabel('hospitals_10km')
plt.ylabel('pharmacies_10km')
plt.grid(alpha=0.3)
plt.show()


### Correlation

In [ ]:
numeric_cols = ['internalFacilitiesCount','hospitals_10km','pharmacies_10km','facilityDiversity_10km','facilityDensity_10km']
correlation = raw_df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation heatmap of key numeric features')
plt.show()

corr_with_diversity = correlation['facilityDiversity_10km'].sort_values(ascending=False)
corr_with_internal = correlation['internalFacilitiesCount'].sort_values(ascending=False)
print('Correlation with facilityDiversity_10km:\n', corr_with_diversity)
print('\nCorrelation with internalFacilitiesCount:\n', corr_with_internal)

# Feature Selection, Preprocessing, and K-Means Clustering

We will use scikit-learn transformers to build a pipeline that standardizes relevant numeric features and fits K-Means. This is purely unsupervised for now (no Gold labels), then we map clusters to `Premium`, `Standard`, `Basic` by center quality signal.

In [ ]:
raw_df.head()

In [ ]:
clean_df = raw_df.copy()

columns_to_drop = ['researchCenterId']
numerical_cols = ['latitude', 'longitude', 'internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km', 'facilityDiversity_10km', 'facilityDensity_10km']
categorical_cols = ['researchCenterName', 'city']

print('Raw rows:', len(raw_df), 'Clean rows:', len(clean_df))

preprocessor = ColumnTransformer(
    transformers=[
        ('remove_columns', 'drop', columns_to_drop),
        ('numerical', StandardScaler(), numerical_cols),
        ('encode', OneHotEncoder(), categorical_cols)
    ],
    remainder='passthrough'
)

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('kmeans', KMeans(n_clusters=3, random_state=42))
])

pipe.fit(clean_df)



In [ ]:
# Save trained pipeline
joblib.dump(pipe, 'kmeans_research_center_pipeline.joblib')
print('Pipeline saved to kmeans_research_center_pipeline.joblib')


## Analysis